In [1]:
import logging

# Custom filter to show only AdditionSubRepository debug messages
class AdditionSubRepositoryFilter(logging.Filter):
    def filter(self, record):
        # Only allow messages that are from AdditionSubRepository methods
        addition_repo_messages = [
            "Building composed repo.",
            "__enter__", 
            "__exit__"
        ]
        return any(msg in record.getMessage() for msg in addition_repo_messages)

# Configure logger to show only AdditionSubRepository debug messages
logging.basicConfig(level=logging.DEBUG)
logging.basicConfig(level=logging.CRITICAL, handlers=[])  # No default handlers
qsub_logger = logging.getLogger('quri_parts.qsub.resolve')
qsub_logger.setLevel(logging.DEBUG)

# Clear any existing handlers
qsub_logger.handlers.clear()

# Add our custom filtered handler
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.addFilter(AdditionSubRepositoryFilter())
formatter = logging.Formatter('%(levelname)s:%(name)s:%(message)s')
handler.setFormatter(formatter)
qsub_logger.addHandler(handler)

# Prevent propagation to root logger
qsub_logger.propagate = False

As an example, let's define a new fake `Op` to elucidate the problem. This new Op:

1. Is only used in some special context, so we don't want to register it to the `default_repository`.
2. Is registered to `addition_repo`, which only holds the resolver of this Op.
3. Is registered to `context_repo`, which is a copy of the `default_repo`


In [2]:
from quri_parts.qsub.resolve import SubRepository, default_repository, resolve_sub
from quri_parts.qsub.opsub import OpSubDef, opsub
from quri_parts.qsub.op import Op, op as op_def, OpDef
from quri_parts.qsub.sub import SubBuilder, Sub
from quri_parts.qsub.lib import std
from quri_parts.qsub.compile import compile_sub
from quri_parts.qsub.primitive import AllBasicSet
from quri_parts.qsub.evaluate import Evaluator
from quri_parts.qsub.eval import QURIPartsEvaluatorHooks

class _TestO1(OpDef):
    name = "TestO1"
    qubit_count = 1


TestO1 = op_def(_TestO1)


def test_o1_alt_resolver(op: Op, repo: SubRepository) -> Sub:
    builder = SubBuilder(1)
    builder.add_op(std.RX(0.123123123), builder.qubits)
    return builder.build()

builder = SubBuilder(2)
builder.add_op(std.PauliRotation((1, 1), 0.87), builder.qubits)
builder.add_op(TestO1, (builder.qubits[1],))

qp_converter = Evaluator(QURIPartsEvaluatorHooks())

Let's compile the sub with the `default_repo`. It would fail as expected, because the resolver is not registered .to the `default_repo`.

In [3]:
try:
    compile_sub(builder.build(), AllBasicSet)
except ValueError as e:
    print(e)

Op __default__.TestO1(qubits=1, registers=0) is not found in calltable.


Let's compile the sub with the `addition_repo`. It also fails because `PauliRotation` is not inside the addition_repo.

In [4]:
addition_repo = SubRepository()
addition_repo.register_sub_resolver(TestO1, test_o1_alt_resolver)


try:
    compile_sub(builder.build(), AllBasicSet, addition_repo)
except ValueError as e:
    print(e)

Op lib.std.PauliRotation<(1, 1), 0.87>(qubits=2, registers=0) is not found in calltable.


Let's compile it agiain with the `context_repo`.

In [5]:
context_repo = default_repository().copy()
context_repo.register_sub_resolver(TestO1, test_o1_alt_resolver)

[o[0].op.base_id[1] for o in compile_sub(builder.build(), AllBasicSet, context_repo).instructions]

['PauliRotation', 'TestO1']

It works! Let's now see what happens when we register a new subresolver to the `default_repo`.

In [6]:
class _TestO2(OpSubDef):
    name = "TestO2"
    qubit_count = 2

    def sub(self, builder: SubBuilder) -> None:
        builder.add_op(std.PauliRotation((1, 1), 0.345), builder.qubits)

TestO2, _ = opsub(_TestO2)

Let's make a Sub with both Ops and compile again.

In [7]:
builder = SubBuilder(2)
builder.add_op(std.PauliRotation((1, 1), 0.87), builder.qubits)
builder.add_op(TestO1, (builder.qubits[1],))
builder.add_op(TestO2, builder.qubits)

try:
    compile_sub(builder.build(), AllBasicSet, context_repo)
except ValueError as e:
    print(e)

Op __default__.TestO2(qubits=2, registers=0) is not found in calltable.


The compilation fails! This is because `context_repo` knows nothing about `TestOp2` when it is made. The only way is to make the context repo again by copying the default, register the resolver for `TestOp1`. This is very cumbersome as users need to keep track of when they need to make the context op.

We introduce a `chain` method for adding the resolvers of the `addition_repo` into default repo. This generates a new type of repository that allows users to chain the `default_repo` with the `addition_repo` and make a new repo that always "knows" about the new resolvers registered into the `default_repo` after the chaining operation.

In [8]:
new_repo = default_repository().chain([addition_repo])

msub = compile_sub(builder.build(), AllBasicSet, new_repo)
qp_converter.run(msub).gates

DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.


(QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RZ', target_indices=(0,), control_indices=(), classical_indices=(), params=(0.87,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RX', target_indices=(1,), control_indices=(), cl

The compilation works as expected. There are multiple logs saying "Building composed repo". We come back to this point later. Now, let's register another Op to the default repo and check if things work as expected with the standard interface.

In [9]:
class _TestO3(OpSubDef):
    name = "TestO3"
    qubit_count = 2

    def sub(self, builder: SubBuilder) -> None:
        builder.add_op(std.Pauli((1, 1)), builder.qubits)

TestO3, _ = opsub(_TestO3)

We do not need to do any `copy` or `chain` again.

In [10]:
builder = SubBuilder(2)
builder.add_op(TestO1, (builder.qubits[1],))
builder.add_op(TestO2, builder.qubits)
builder.add_op(TestO3, builder.qubits)

msub = compile_sub(builder.build(), AllBasicSet, new_repo)
qp_converter.run(msub).gates

DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.


(QuantumGate(name='RX', target_indices=(1,), control_indices=(), classical_indices=(), params=(0.123123123,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RZ', target_indices=(0,), control_indices=(), classical_indices=(), params=(0.345,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_i

It indeed works and the 2 X gate at the end of the circuit shows that the sub gets compiled into the proper circuit.

## Introducing the new object `CompositeSubRepository`

What's really happening under the hood is that calling `default_repository.chain()` creates a `CompositeSubRepository`. This is the same as creating one in the following way.

In [11]:
from quri_parts.qsub.resolve import CompositeSubRepository 

new_repo = CompositeSubRepository(additions=[addition_repo], root_repo=default_repository())

msub = compile_sub(builder.build(), AllBasicSet, new_repo)
qp_converter.run(msub).gates

DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.


(QuantumGate(name='RX', target_indices=(1,), control_indices=(), classical_indices=(), params=(0.123123123,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RZ', target_indices=(0,), control_indices=(), classical_indices=(), params=(0.345,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_i

What happens under the hood is that `CompositeSubRepository` keeps a reference to the `default_repository` the `default_repository` gets into a new temporary repository for the additions gets registered to it. This is why there are logging messages saying "Building composed repo". 

Having multiple "Building composed repo" logging messages is because the above operation is done repeated once `find_resolver` of the `CompositeSubRepository` is called. This is very inefficient when compiling large Subs. So we supply a context manager so the above operation is done once.

In [12]:
with new_repo:
    msub = compile_sub(builder.build(), AllBasicSet, new_repo)

qp_converter.run(msub).gates

DEBUG:quri_parts.qsub.resolve:__enter__
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:__exit__


(QuantumGate(name='RX', target_indices=(1,), control_indices=(), classical_indices=(), params=(0.123123123,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RZ', target_indices=(0,), control_indices=(), classical_indices=(), params=(0.345,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_i

Another way of doing this is to call the `flatten` method.

In [13]:
msub = compile_sub(builder.build(), AllBasicSet, new_repo.flatten())
qp_converter.run(msub).gates

DEBUG:quri_parts.qsub.resolve:Building composed repo.


(QuantumGate(name='RX', target_indices=(1,), control_indices=(), classical_indices=(), params=(0.123123123,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='RZ', target_indices=(0,), control_indices=(), classical_indices=(), params=(0.345,), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='CNOT', target_indices=(0,), control_indices=(1,), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(0,), control_indices=(), classical_indices=(), params=(), pauli_ids=(), unitary_matrix=()),
 QuantumGate(name='H', target_indices=(1,), control_i

(I actually don't know if we really need the context manager at all as `flatten` can do the exact same thing. This comes with a little bit of memory overhead though.)

## Overriding in action

Now, let's see how to overriding works in action. A good example is the multi-controlled feature. The resolvers for resolving multi-controlled Op into multi-controlled gate set needs to be actively registered into a SubRepository for the Sub to be compiled into the `SimulatorBasicSet`. 


We provide the `simulator_repository` for this purpose.

In [14]:
from typing import Iterable
from quri_parts.qsub.op import AbstractOp
from quri_parts.qsub.op import BaseIdent
from quri_parts.qsub.primitive import SimulatorBasicSet
from quri_parts.qsub.eval import GateCountEvaluatorHooks

def get_name_count(gate_count: dict[BaseIdent, int]) -> dict[str, int]:
    return {g[1]: v for g, v in gate_count.items()}

gate_counter = Evaluator(GateCountEvaluatorHooks())

builder = SubBuilder(3)
builder.add_op(std.MultiControlled(std.H, 2, 0b11), builder.qubits)


def get_sub_gate_count(
    sub: Sub, primitives: Iterable[AbstractOp], repo: SubRepository = default_repository()
) -> dict[str, int]:
    return get_name_count(
        gate_counter.run(compile_sub(sub, primitives, repo))
    )

Use `default_repository` and compile with `SimulatorBasicSet`.

In [15]:
get_sub_gate_count(builder.build(), SimulatorBasicSet, default_repository())

{'Toffoli': 2, 'RY': 2, 'CZ': 1}

Multi-control-basis-gate resolver does not exist in the `default_repo`, thus this returns primitives. We now use `simulator_repository` and compile the sub into `SimulatorBasicSet`.

In [16]:
from quri_parts.qsub.resolve import simulator_repository

get_sub_gate_count(builder.build(), SimulatorBasicSet, simulator_repository())

DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:Building composed repo.


{'MCH': 1}

Now we get MCH gate in return. As we shown above, we need to use the composite repository with either context manager or `flatten` method to have optimal performance.

In [17]:
get_sub_gate_count(builder.build(), SimulatorBasicSet, simulator_repository().flatten())

DEBUG:quri_parts.qsub.resolve:Building composed repo.


{'MCH': 1}

This indeed gives the correct compilation result with the performance

We also check compiling a sub into `AllBasiSet` using `simulator_repository`.

In [18]:
get_sub_gate_count(builder.build(), AllBasicSet, simulator_repository().flatten())

DEBUG:quri_parts.qsub.resolve:Building composed repo.


{'Toffoli': 2, 'RY': 2, 'CZ': 1}

Finally, we check that the `simulator_repository` works as intended after registering a new resolver into the default repository.

In [19]:
class _MCHWrapper(OpSubDef):
    name = "MCHWrapper"
    qubit_count = 3

    def sub(self, builder: SubBuilder) -> None:
        builder.add_op(std.MultiControlled(std.H, 2, 0b11), builder.qubits)

MCHWrapper, _ = opsub(_MCHWrapper)

In [20]:
with simulator_repository() as repo:
    count = get_sub_gate_count(resolve_sub(MCHWrapper, repo), SimulatorBasicSet, repo)

count

DEBUG:quri_parts.qsub.resolve:__enter__
DEBUG:quri_parts.qsub.resolve:Building composed repo.
DEBUG:quri_parts.qsub.resolve:__exit__


{'MCH': 1}

## Final note

The idea behind this design is that we assume the `default_repository` as the "root_repository" where the composite repository bases on. A good practice is to register all Ops into the default repository so all `CompositeSubRepository`s know about it and determine how to resolve it base whether there are additions or not. 

In summary there are 3 types of sub repositories

* Type 1: The root repo, in most cases is the `default_repository`, which knows about all Ops and their default resolver. This is represented by a `SubRepository`.
* Type 2: Addition repositories that stores additional resolvers, also represented by a `SubRepository`.
* Type 3: `CompositeRepository` that dynamically joins `default_repository` and its addition. Its relation with the other repos are summarized as the following:
    * `default_repo.chain([addition_repos])` -> CompositeSubRepo
    * `composite_repo.chain([addition_repos])` -> CompositeSubRepo (`default_repo` set as the root of the new composite repo, `composite_repo.additions` + `addition_repos` becomes the addition of the new repo.)
    * `composite_repo.flatten()` creates a temporary repo for resolving sub. It is called every time `composite_repo.find_resolver` is called, unless `composite_repo` called as a context manager.
